# Unit 13 — Grids, Graphs & Traversal

What is the fewest number of steps through a maze? How many separate rooms are connected? Both questions become easier when we treat each open location as a node and each allowed move as an edge. In this unit, we will represent those connections and traverse them without getting stuck in cycles.

## Lesson 1 — Four Neighbours in a Grid

A cell at row `r`, column `c` can have four neighbours: up, down, left, and right. Diagonal cells are not neighbours. For every candidate `(nr, nc)`, check both coordinates before reading the grid: `0 <= nr and nr < rows` and `0 <= nc and nc < cols`. Separate comparisons make every boundary explicit.

The demo counts the open neighbours marked `.` around one requested cell. Corners have fewer possible neighbours than interior cells.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    rows = int(tokens[0])
    cols = int(tokens[1])
    grid = []
    i = 0
    while i < rows:
        grid.append(tokens[i + 2])
        i = i + 1
    r = int(tokens[rows + 2])
    c = int(tokens[rows + 3])
    dr = [-1, 1, 0, 0]
    dc = [0, 0, -1, 1]
    count = 0
    direction = 0
    while direction < 4:
        nr = r + dr[direction]
        nc = c + dc[direction]
        if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
            if grid[nr][nc] == ".":
                count = count + 1
        direction = direction + 1
    return str(count)

assert solve("3 4 .... .##. .... 1 0") == "2"
assert solve("2 2 .. .# 0 0") == "2"

## An Adjacency-List Graph

A graph has **nodes** and **edges**. Store a graph in a plain dictionary whose key is a node and whose value is a list of its neighbours: `{node: [neighbours]}`. Before appending an endpoint, create its empty list when it is not already a key.

For an undirected edge `u v`, append `v` to `adj[u]` **and** append `u` to `adj[v]`. The degree of `node` is `len(adj[node])`. Adding only one direction would describe a different graph and give the wrong degree at the other endpoint.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    edges = int(tokens[0])
    adj = {}
    position = 1
    i = 0
    while i < edges:
        u = int(tokens[position])
        v = int(tokens[position + 1])
        if u not in adj:
            adj[u] = []
        if v not in adj:
            adj[v] = []
        adj[u].append(v)
        adj[v].append(u)
        position = position + 2
        i = i + 1
    node = int(tokens[position])
    return str(len(adj[node]))

assert solve("4 1 2 2 3 2 4 4 5 2") == "3"
assert solve("2 7 8 8 9 9") == "1"

## Lesson 2 — Recursive Flood-Fill

A **flood-fill** starts at one grid cell and visits its whole connected region. The recursive function receives the grid position and a `visited` set as arguments. If the position is outside the grid, blocked, or already visited, it stops. Otherwise it adds `(r, c)` to `visited` and recursively explores the four neighbours.

Passing and mutating the same set lets every recursive call share what has already been explored. It also prevents a call from immediately returning to its parent forever. Recursive grid problems in this course keep `rows * cols <= 400`.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    rows = int(tokens[0])
    cols = int(tokens[1])
    grid = []
    i = 0
    while i < rows:
        grid.append(tokens[i + 2])
        i = i + 1
    start_r = int(tokens[rows + 2])
    start_c = int(tokens[rows + 3])
    visited = set()

    def fill(r, c, visited):
        if not (0 <= r and r < rows and 0 <= c and c < cols):
            return
        if grid[r][c] == "#" or (r, c) in visited:
            return
        visited.add((r, c))
        fill(r - 1, c, visited)
        fill(r + 1, c, visited)
        fill(r, c - 1, visited)
        fill(r, c + 1, visited)

    fill(start_r, start_c, visited)
    return str(len(visited))

assert solve("3 4 ..#. .##. .... 0 0") == "9"
assert solve("2 2 .# #. 0 0") == "1"

## Breadth-First Search Finds Fewest Steps

**Breadth-first search**, or **BFS**, explores all positions at distance `0`, then distance `1`, then distance `2`, and so on. That layer order makes the first visit to a cell its shortest distance when every move costs one step.

Use a `deque` as a FIFO queue: `append` to enqueue at the back and `popleft` to dequeue from the front. Add a cell to `visited` when it is enqueued, not later, so it enters the queue only once. A distance dictionary records `distance[next_cell] = distance[current] + 1`. Return `-1` when the target is unreachable.

In [ ]:
from collections import deque

def solve(data: str) -> str:
    tokens = data.split()
    rows = int(tokens[0])
    cols = int(tokens[1])
    grid = []
    start = (-1, -1)
    target = (-1, -1)
    r = 0
    while r < rows:
        row = tokens[r + 2]
        grid.append(row)
        c = 0
        while c < cols:
            if row[c] == "S":
                start = (r, c)
            elif row[c] == "T":
                target = (r, c)
            c = c + 1
        r = r + 1
    queue = deque()
    queue.append(start)
    visited = {start}
    distance = {start: 0}
    dr = [-1, 1, 0, 0]
    dc = [0, 0, -1, 1]
    while len(queue) > 0:
        current = queue.popleft()
        if current == target:
            return str(distance[current])
        direction = 0
        while direction < 4:
            nr = current[0] + dr[direction]
            nc = current[1] + dc[direction]
            if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
                next_cell = (nr, nc)
                if grid[nr][nc] != "#" and next_cell not in visited:
                    visited.add(next_cell)
                    distance[next_cell] = distance[current] + 1
                    queue.append(next_cell)
            direction = direction + 1
    return "-1"

assert solve("4 5 S...# .##.# ...#. #...T") == "7"
assert solve("3 3 S#. ### .#T") == "-1"

## Lesson 3 — Depth-First Search Reachability

**Depth-first search**, or **DFS**, follows one route as far as it can before trying another. For a reachability question, a recursive DFS marks the current node, checks its neighbours, and recursively visits each unvisited neighbour.

The `visited` set must be passed as an argument and mutated with `.add`. It prevents cycles such as `1-2-3-1` from causing infinite recursion. BFS is the right tool for a shortest unweighted path; DFS is enough when the question only asks whether some path exists.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    edges = int(tokens[0])
    adj = {}
    position = 1
    i = 0
    while i < edges:
        u = int(tokens[position])
        v = int(tokens[position + 1])
        if u not in adj:
            adj[u] = []
        if v not in adj:
            adj[v] = []
        adj[u].append(v)
        adj[v].append(u)
        position = position + 2
        i = i + 1
    start = int(tokens[position])
    target = int(tokens[position + 1])
    visited = set()

    def reaches(node, target, visited):
        if node == target:
            return True
        visited.add(node)
        neighbours = adj.get(node, [])
        i = 0
        while i < len(neighbours):
            next_node = neighbours[i]
            if next_node not in visited:
                if reaches(next_node, target, visited):
                    return True
            i = i + 1
        return False

    if reaches(start, target, visited):
        return "YES"
    return "NO"

assert solve("4 1 2 2 3 3 1 3 4 1 4") == "YES"
assert solve("3 1 2 2 3 4 5 1 5") == "NO"

## Traversal Checklist

For a grid, list only four directions and check row and column bounds before indexing. For an undirected graph, store every edge at both endpoints. Mark a node or cell visited before exploring onward. Use BFS with a FIFO deque for fewest steps, recursive DFS for reachability, and recursive flood-fill for an entire grid region. Pass the shared `visited` set as an argument.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper. It is marked `no-exec` because notebook checks call `solve` directly.

In [ ]:
import sys
print(solve(sys.stdin.read()))